# Sioux Falls: TAP from DataFrames

Build the network and OD demand as pandas DataFrames, solve user equilibrium,
check the assignment, and inspect congestion. Then compare TAP approaches and
run a demand scenario.

**Setup:** from the repository root, run `pip install -e '.[examples]'`,
then `jupyter lab examples/`. Select a Python kernel with this package installed
and run all cells in order. This notebook is independent of the CNDP notebook.

## 1. Create the input DataFrames

The complete benchmark values are embedded below: **24 nodes, 24 zones,
76 directed links, and total demand 360,600**. No CSV files or network
downloads are needed. These values are the project's Sioux Falls input,
sourced from [Transportation Networks for Research](https://github.com/bstabler/TransportationNetworks/tree/master/SiouxFalls).
We retain the benchmark's numeric units throughout; time values are not
relabelled as minutes.

Each link uses the BPR function
$t(f) = t_0[1 + b(f/c)^p]$. Nodes and demand labels are **one-based**.
`link_index` is **zero-based**, identifies a directed link, and remains
attached to that link if a DataFrame is sorted.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import traffic_assignment as ta

pd.set_option("display.max_columns", 16)
pd.set_option("display.precision", 4)

In [ ]:
# One row per directed link; all values are benchmark inputs.
links_df = pd.DataFrame(
    [[1, 2, 25900.20064, 6, 6, 0.15, 4, 0, 0, 1],
     [1, 3, 23403.47319, 4, 4, 0.15, 4, 0, 0, 1],
     [2, 1, 25900.20064, 6, 6, 0.15, 4, 0, 0, 1],
     [2, 6, 4958.180928, 5, 5, 0.15, 4, 0, 0, 1],
     [3, 1, 23403.47319, 4, 4, 0.15, 4, 0, 0, 1],
     [3, 4, 17110.52372, 4, 4, 0.15, 4, 0, 0, 1],
     [3, 12, 23403.47319, 4, 4, 0.15, 4, 0, 0, 1],
     [4, 3, 17110.52372, 4, 4, 0.15, 4, 0, 0, 1],
     [4, 5, 17782.7941, 2, 2, 0.15, 4, 0, 0, 1],
     [4, 11, 4908.82673, 6, 6, 0.15, 4, 0, 0, 1],
     [5, 4, 17782.7941, 2, 2, 0.15, 4, 0, 0, 1],
     [5, 6, 4947.995469, 4, 4, 0.15, 4, 0, 0, 1],
     [5, 9, 10000.0, 5, 5, 0.15, 4, 0, 0, 1],
     [6, 2, 4958.180928, 5, 5, 0.15, 4, 0, 0, 1],
     [6, 5, 4947.995469, 4, 4, 0.15, 4, 0, 0, 1],
     [6, 8, 4898.587646, 2, 2, 0.15, 4, 0, 0, 1],
     [7, 8, 7841.81131, 3, 3, 0.15, 4, 0, 0, 1],
     [7, 18, 23403.47319, 2, 2, 0.15, 4, 0, 0, 1],
     [8, 6, 4898.587646, 2, 2, 0.15, 4, 0, 0, 1],
     [8, 7, 7841.81131, 3, 3, 0.15, 4, 0, 0, 1],
     [8, 9, 5050.193156, 10, 10, 0.15, 4, 0, 0, 1],
     [8, 16, 5045.822583, 5, 5, 0.15, 4, 0, 0, 1],
     [9, 5, 10000.0, 5, 5, 0.15, 4, 0, 0, 1],
     [9, 8, 5050.193156, 10, 10, 0.15, 4, 0, 0, 1],
     [9, 10, 13915.78842, 3, 3, 0.15, 4, 0, 0, 1],
     [10, 9, 13915.78842, 3, 3, 0.15, 4, 0, 0, 1],
     [10, 11, 10000.0, 5, 5, 0.15, 4, 0, 0, 1],
     [10, 15, 13512.00155, 6, 6, 0.15, 4, 0, 0, 1],
     [10, 16, 4854.917717, 4, 4, 0.15, 4, 0, 0, 1],
     [10, 17, 4993.510694, 8, 8, 0.15, 4, 0, 0, 1],
     [11, 4, 4908.82673, 6, 6, 0.15, 4, 0, 0, 1],
     [11, 10, 10000.0, 5, 5, 0.15, 4, 0, 0, 1],
     [11, 12, 4908.82673, 6, 6, 0.15, 4, 0, 0, 1],
     [11, 14, 4876.508287, 4, 4, 0.15, 4, 0, 0, 1],
     [12, 3, 23403.47319, 4, 4, 0.15, 4, 0, 0, 1],
     [12, 11, 4908.82673, 6, 6, 0.15, 4, 0, 0, 1],
     [12, 13, 25900.20064, 3, 3, 0.15, 4, 0, 0, 1],
     [13, 12, 25900.20064, 3, 3, 0.15, 4, 0, 0, 1],
     [13, 24, 5091.256152, 4, 4, 0.15, 4, 0, 0, 1],
     [14, 11, 4876.508287, 4, 4, 0.15, 4, 0, 0, 1],
     [14, 15, 5127.526119, 5, 5, 0.15, 4, 0, 0, 1],
     [14, 23, 4924.790605, 4, 4, 0.15, 4, 0, 0, 1],
     [15, 10, 13512.00155, 6, 6, 0.15, 4, 0, 0, 1],
     [15, 14, 5127.526119, 5, 5, 0.15, 4, 0, 0, 1],
     [15, 19, 14564.75315, 3, 3, 0.15, 4, 0, 0, 1],
     [15, 22, 9599.180565, 3, 3, 0.15, 4, 0, 0, 1],
     [16, 8, 5045.822583, 5, 5, 0.15, 4, 0, 0, 1],
     [16, 10, 4854.917717, 4, 4, 0.15, 4, 0, 0, 1],
     [16, 17, 5229.910063, 2, 2, 0.15, 4, 0, 0, 1],
     [16, 18, 19679.89671, 3, 3, 0.15, 4, 0, 0, 1],
     [17, 10, 4993.510694, 8, 8, 0.15, 4, 0, 0, 1],
     [17, 16, 5229.910063, 2, 2, 0.15, 4, 0, 0, 1],
     [17, 19, 4823.950831, 2, 2, 0.15, 4, 0, 0, 1],
     [18, 7, 23403.47319, 2, 2, 0.15, 4, 0, 0, 1],
     [18, 16, 19679.89671, 3, 3, 0.15, 4, 0, 0, 1],
     [18, 20, 23403.47319, 4, 4, 0.15, 4, 0, 0, 1],
     [19, 15, 14564.75315, 3, 3, 0.15, 4, 0, 0, 1],
     [19, 17, 4823.950831, 2, 2, 0.15, 4, 0, 0, 1],
     [19, 20, 5002.607563, 4, 4, 0.15, 4, 0, 0, 1],
     [20, 18, 23403.47319, 4, 4, 0.15, 4, 0, 0, 1],
     [20, 19, 5002.607563, 4, 4, 0.15, 4, 0, 0, 1],
     [20, 21, 5059.91234, 6, 6, 0.15, 4, 0, 0, 1],
     [20, 22, 5075.697193, 5, 5, 0.15, 4, 0, 0, 1],
     [21, 20, 5059.91234, 6, 6, 0.15, 4, 0, 0, 1],
     [21, 22, 5229.910063, 2, 2, 0.15, 4, 0, 0, 1],
     [21, 24, 4885.357564, 3, 3, 0.15, 4, 0, 0, 1],
     [22, 15, 9599.180565, 3, 3, 0.15, 4, 0, 0, 1],
     [22, 20, 5075.697193, 5, 5, 0.15, 4, 0, 0, 1],
     [22, 21, 5229.910063, 2, 2, 0.15, 4, 0, 0, 1],
     [22, 23, 5000.0, 4, 4, 0.15, 4, 0, 0, 1],
     [23, 14, 4924.790605, 4, 4, 0.15, 4, 0, 0, 1],
     [23, 22, 5000.0, 4, 4, 0.15, 4, 0, 0, 1],
     [23, 24, 5078.508436, 2, 2, 0.15, 4, 0, 0, 1],
     [24, 13, 5091.256152, 4, 4, 0.15, 4, 0, 0, 1],
     [24, 21, 4885.357564, 3, 3, 0.15, 4, 0, 0, 1],
     [24, 23, 5078.508436, 2, 2, 0.15, 4, 0, 0, 1]],
    columns=['init_node', 'term_node', 'capacity', 'length', 'free_flow_time', 'b', 'power', 'speed', 'toll', 'link_type'],
)
links_df.insert(0, "link_index", np.arange(len(links_df)))
display(links_df.head())

In [ ]:
# Rows are origins, columns are destinations.
zone_ids = range(1, 25)
demand_df = pd.DataFrame(
    [[0, 100, 100, 500, 200, 300, 500, 800, 500, 1300, 500, 200, 500, 300, 500, 500, 400, 100, 300, 300, 100, 400, 300, 100],
     [100, 0, 100, 200, 100, 400, 200, 400, 200, 600, 200, 100, 300, 100, 100, 400, 200, 0, 100, 100, 0, 100, 0, 0],
     [100, 100, 0, 200, 100, 300, 100, 200, 100, 300, 300, 200, 100, 100, 100, 200, 100, 0, 0, 0, 0, 100, 100, 0],
     [500, 200, 200, 0, 500, 400, 400, 700, 700, 1200, 1400, 600, 600, 500, 500, 800, 500, 100, 200, 300, 200, 400, 500, 200],
     [200, 100, 100, 500, 0, 200, 200, 500, 800, 1000, 500, 200, 200, 100, 200, 500, 200, 0, 100, 100, 100, 200, 100, 0],
     [300, 400, 300, 400, 200, 0, 400, 800, 400, 800, 400, 200, 200, 100, 200, 900, 500, 100, 200, 300, 100, 200, 100, 100],
     [500, 200, 100, 400, 200, 400, 0, 1000, 600, 1900, 500, 700, 400, 200, 500, 1400, 1000, 200, 400, 500, 200, 500, 200, 100],
     [800, 400, 200, 700, 500, 800, 1000, 0, 800, 1600, 800, 600, 600, 400, 600, 2200, 1400, 300, 700, 900, 400, 500, 300, 200],
     [500, 200, 100, 700, 800, 400, 600, 800, 0, 2800, 1400, 600, 600, 600, 900, 1400, 900, 200, 400, 600, 300, 700, 500, 200],
     [1300, 600, 300, 1200, 1000, 800, 1900, 1600, 2800, 0, 4000, 2000, 1900, 2100, 4000, 4400, 3900, 700, 1800, 2500, 1200, 2600, 1800, 800],
     [500, 200, 300, 1500, 500, 400, 500, 800, 1400, 3900, 0, 1400, 1000, 1600, 1400, 1400, 1000, 100, 400, 600, 400, 1100, 1300, 600],
     [200, 100, 200, 600, 200, 200, 700, 600, 600, 2000, 1400, 0, 1300, 700, 700, 700, 600, 200, 300, 400, 300, 700, 700, 500],
     [500, 300, 100, 600, 200, 200, 400, 600, 600, 1900, 1000, 1300, 0, 600, 700, 600, 500, 100, 300, 600, 600, 1300, 800, 800],
     [300, 100, 100, 500, 100, 100, 200, 400, 600, 2100, 1600, 700, 600, 0, 1300, 700, 700, 100, 300, 500, 400, 1200, 1100, 400],
     [500, 100, 100, 500, 200, 200, 500, 600, 1000, 4000, 1400, 700, 700, 1300, 0, 1200, 1500, 200, 800, 1100, 800, 2600, 1000, 400],
     [500, 400, 200, 800, 500, 900, 1400, 2200, 1400, 4400, 1400, 700, 600, 700, 1200, 0, 2800, 500, 1300, 1600, 600, 1200, 500, 300],
     [400, 200, 100, 500, 200, 500, 1000, 1400, 900, 3900, 1000, 600, 500, 700, 1500, 2800, 0, 600, 1700, 1700, 600, 1700, 600, 300],
     [100, 0, 0, 100, 0, 100, 200, 300, 200, 700, 200, 200, 100, 100, 200, 500, 600, 0, 300, 400, 100, 300, 100, 0],
     [300, 100, 0, 200, 100, 200, 400, 700, 400, 1800, 400, 300, 300, 300, 800, 1300, 1700, 300, 0, 1200, 400, 1200, 300, 100],
     [300, 100, 0, 300, 100, 300, 500, 900, 600, 2500, 600, 500, 600, 500, 1100, 1600, 1700, 400, 1200, 0, 1200, 2400, 700, 400],
     [100, 0, 0, 200, 100, 100, 200, 400, 300, 1200, 400, 300, 600, 400, 800, 600, 600, 100, 400, 1200, 0, 1800, 700, 500],
     [400, 100, 100, 400, 200, 200, 500, 500, 700, 2600, 1100, 700, 1300, 1200, 2600, 1200, 1700, 300, 1200, 2400, 1800, 0, 2100, 1100],
     [300, 0, 100, 500, 100, 100, 200, 300, 500, 1800, 1300, 700, 800, 1100, 1000, 500, 600, 100, 300, 700, 700, 2100, 0, 700],
     [100, 0, 0, 200, 0, 100, 100, 200, 200, 800, 600, 500, 700, 400, 400, 300, 300, 0, 100, 400, 500, 1100, 700, 0]],
    index=pd.Index(zone_ids, name="origin"),
    columns=pd.Index(zone_ids, name="destination"),
    dtype=float,
)
assert len(links_df) == 76 and demand_df.shape == (24, 24)
assert demand_df.to_numpy().sum() == 360600.0
display(demand_df.iloc[:6, :6])

## 2. Build a native network and solve TAP

The DataFrame adapter validates the columns and aligns demand by zone label.
`node_index_base=1` converts the benchmark IDs to native zero-based IDs.
The native network owns its input values; later DataFrame edits do not change it.
TAP updates the network's assignment state and returns a snapshot of results.

In [ ]:
network = ta.network_from_dataframes(
    "SiouxFalls", links_df, demand_df, node_index_base=1,
)
tap_options = ta.TapOptions()
tap_options.approach = "tapas"
tap_options.relative_gap_tolerance = 1e-10

result = ta.solve_tap(network, approach=tap_options)
tap_summary = pd.DataFrame([{
    "approach": result.approach,
    "nodes": network.number_of_nodes,
    "zones": network.number_of_zones,
    "links": network.number_of_links,
    "relative_gap": result.relative_gap,
    "total_travel_time": result.total_travel_time,
    "beckmann_objective": result.beckmann_objective,
    "solve_seconds": result.solve_seconds,
}])
display(tap_summary)

## 3. Join flows back to links and check the assignment

Result arrays follow native `link_index` order. Join by that key so analysis
stays correct even when link rows are sorted. `flow / capacity` measures
congestion; capacity is a BPR parameter, so equilibrium flows may exceed it.
The checks below cover nonnegative flows, the travel-time total, convergence,
and aggregate flow conservation at each node.

In [ ]:
flow_df = pd.DataFrame({
    "link_index": np.arange(network.number_of_links),
    "flow": result.flows,
    "travel_time": result.link_costs,
})
link_results = links_df.merge(flow_df, on="link_index", validate="one_to_one")
link_results["volume_capacity_ratio"] = link_results["flow"] / link_results["capacity"]
link_results["travel_time_ratio"] = link_results["travel_time"] / link_results["free_flow_time"]
link_results["total_travel_time"] = link_results["flow"] * link_results["travel_time"]
link_results["arc"] = (
    link_results["init_node"].astype(str) + " → " + link_results["term_node"].astype(str)
)

assert result.relative_gap < 1e-8
assert np.all(result.flows >= -1e-9)
np.testing.assert_allclose(
    link_results["total_travel_time"].sum(), result.total_travel_time, rtol=1e-10,
)
outgoing = np.bincount(link_results.init_node - 1, weights=link_results.flow, minlength=24)
incoming = np.bincount(link_results.term_node - 1, weights=link_results.flow, minlength=24)
expected_balance = demand_df.sum(axis=1).to_numpy() - demand_df.sum(axis=0).to_numpy()
np.testing.assert_allclose(outgoing - incoming, expected_balance, atol=1e-5)
display(link_results.nlargest(12, "volume_capacity_ratio")[[
    "link_index", "arc", "capacity", "flow", "travel_time", "volume_capacity_ratio",
]])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
busiest = link_results.nlargest(15, "volume_capacity_ratio").sort_values("volume_capacity_ratio")
axes[0].barh(busiest["arc"], busiest["volume_capacity_ratio"], color="#2878a5")
axes[0].axvline(1.0, color="#bd4b35", linestyle="--", label="Flow equals capacity")
axes[0].set(xlabel="Flow / capacity", ylabel="Directed link", title="Most congested links")
axes[0].legend()
heatmap = axes[1].imshow(demand_df.to_numpy(), cmap="Blues", origin="upper")
axes[1].set_xticks([0, 5, 11, 17, 23], labels=[1, 6, 12, 18, 24])
axes[1].set_yticks([0, 5, 11, 17, 23], labels=[1, 6, 12, 18, 24])
axes[1].set(xlabel="Destination zone", ylabel="Origin zone", title="OD demand")
fig.colorbar(heatmap, ax=axes[1], label="Demand (benchmark units)")
plt.show()

## 4. Compare the two TAP approaches

Use a fresh network for the route-based solve. Compare aggregate travel time
and the relative gap; different algorithms can have different convergence
paths and runtimes. This single execution is a workflow check, not a timing study.

In [ ]:
route_network = ta.network_from_dataframes(
    "SiouxFalls", links_df, demand_df, node_index_base=1,
)
route_result = ta.solve_tap(
    route_network, approach="routebased", relative_gap_tolerance=1e-10,
    max_iterations=200,
)
approach_comparison = pd.DataFrame([
    {
        "approach": item.approach,
        "relative_gap": item.relative_gap,
        "total_travel_time": item.total_travel_time,
        "solve_seconds": item.solve_seconds,
    }
    for item in (result, route_result)
])
assert route_result.relative_gap < 1e-6
np.testing.assert_allclose(route_result.total_travel_time, result.total_travel_time, rtol=1e-4)
display(approach_comparison)

## 5. Run a demand scenario in memory

Increase all OD demands by 20%, build a separate network, and solve again.
The original DataFrames and the baseline result stay available for comparison.
Edit `demand_multiplier` to explore another scenario.

In [ ]:
demand_multiplier = 1.20
scenario_demand_df = demand_df * demand_multiplier
scenario_network = ta.network_from_dataframes(
    "SiouxFalls-demand-scenario", links_df, scenario_demand_df, node_index_base=1,
)
scenario = ta.solve_tap(scenario_network, approach=tap_options)
scenario_comparison = pd.DataFrame({
    "scenario": ["Baseline", f"Demand × {demand_multiplier:.2f}"],
    "total_demand": [demand_df.to_numpy().sum(), scenario_demand_df.to_numpy().sum()],
    "total_travel_time": [result.total_travel_time, scenario.total_travel_time],
    "relative_gap": [result.relative_gap, scenario.relative_gap],
})
assert scenario.relative_gap < 1e-8
assert demand_df.to_numpy().sum() == 360600.0
display(scenario_comparison)